# 🚦 Traffic Accident Hotspot Detection and Severity Prediction
## Notebook 04: Feature Engineering

**Using Geospatial Clustering and Machine Learning**

---

| | |
|---|---|
| **Dataset** | US Accidents (2016–2023), preprocessed 300,000-row sample |
| **Environment** | Google Colab |
| **Notebook Role** | Feature Engineering ONLY (no clustering, no model training) |
| **Pipeline Position** | 4 of 6 |
| **Depends on** | `processed_accidents.csv` from Notebook 03 — the *only* input this notebook loads |

---

## 🎯 Objective

Notebook 03 handed us a single clean, correctly-typed dataset — no train/test split, no scaling, no high-cardinality encoding, all deferred to Notebook 06. This notebook's job is narrower and purely additive:

> "Turn the cleaned dataset into a **richer** dataset by engineering new, meaningful features — without performing any machine learning, clustering, splitting, or scaling."

**Strict scope for this notebook:**
- ✅ Load `processed_accidents.csv` — nothing else
- ✅ Temporal features derived from `Start_Time` / `End_Time`
- ✅ Weather-derived risk flags and a composite weather severity score
- ✅ Road-infrastructure composite features
- ✅ Geographic helper features to prepare for DBSCAN (Notebook 05) — **no clustering here**
- ✅ Interaction features between temporal, weather, and road signals
- ✅ Feature selection, a Feature Dictionary, and a correlation/usefulness analysis, with every decision justified
- ✅ Exporting `engineered_accidents.csv` and `feature_list.json` for Notebook 05

**What this notebook deliberately does NOT do:**
- ❌ No train/test split (→ Notebook 06)
- ❌ No feature scaling (→ Notebook 06)
- ❌ No frequency encoding of `City`/`State`/`Weather_Condition`/`Wind_Direction` (→ Notebook 06)
- ❌ No DBSCAN clustering (→ Notebook 05)
- ❌ No Random Forest / XGBoost training (→ Notebook 06)
- ❌ No model evaluation or hyperparameter tuning (→ later notebooks)

**A rule we hold ourselves to throughout:** every engineered feature must be traceable to a finding from Notebook 01 (structure), Notebook 02 (EDA insight), or Notebook 03 (what shape the data is actually in). Nothing here is invented for the sake of having more columns.


## 📑 Table of Contents

1. [Introduction](#introduction)
2. [Load Dataset](#load-dataset)
3. [Existing Feature Review](#existing-feature-review)
4. [Temporal Feature Engineering](#temporal-feature-engineering)
5. [Weather Feature Engineering](#weather-feature-engineering)
6. [Road Feature Engineering](#road-feature-engineering)
7. [Geographic Feature Engineering](#geographic-feature-engineering)
8. [Interaction Features](#interaction-features)
9. [Feature Selection](#feature-selection)
10. [Feature Dictionary](#feature-dictionary)
11. [Correlation & Feature Usefulness Analysis](#correlation-analysis)
12. [Final Dataset](#final-dataset)
13. [Export engineered_accidents.csv and feature_list.json](#export)
14. [Notebook Summary](#notebook-summary)
15. [Preparing for Notebook 5](#next-notebook-preview)


## 1. Introduction <a name="introduction"></a>

**Where we are in the pipeline:**

```
Raw Dataset
      ↓
Dataset Understanding (01)
      ↓
Exploratory Data Analysis (02)
      ↓
Data Preprocessing (03)  →  processed_accidents.csv (single clean dataset, no split)
      ↓
Feature Engineering (04)  ← WE ARE HERE  →  engineered_accidents.csv, feature_list.json
      ↓
DBSCAN Hotspot Detection (05)  →  dataset_with_hotspots.csv
      ↓
Severity Prediction (06)  →  train/test split, scaling, frequency encoding, modeling
```

**A design principle we carry forward from Notebook 03, stated explicitly:** in this architecture, the train/test split does not happen until Notebook 06 — immediately before modeling. That means Notebooks 03, 04, and 05 all necessarily compute any dataset-level statistic (an IQR bound, a percentile, a grid-cell count) on the **full** dataset, simply because no split exists yet at this stage. This is the same convention Notebook 03 used when it capped outliers on the full dataset rather than a training-only subset.

**Why this is a reasonable design, not a leakage shortcut:** the statistics computed in this notebook (visibility thresholds, grid densities, IQR bounds) never use the target variable (`Severity`), so no information about *what we're trying to predict* leaks anywhere. What they *do* carry is a mild, well-known form of exposure — a threshold computed over rows that will later become Notebook 06's test set. For non-target statistics used only to build interpretable flags, this is a standard, widely-accepted simplification in projects that separate feature engineering from modeling this way — but it is a real design tradeoff, not a non-issue, so we name it here rather than leaving it implicit. Wherever practical below, we additionally prefer **fixed, domain-motivated thresholds** (e.g., "visibility below 1 mile") over data-derived percentiles, specifically because a domain threshold doesn't depend on this dataset's distribution at all, sidestepping the concern entirely.

**Row-wise transformations carry no such exposure:** features that are pure, per-row transformations of existing columns (extracting the hour from a timestamp, string-matching a weather description) depend only on that single row's own values — there is nothing here for any other row, train or test, to leak.


## 2. Load Dataset <a name="load-dataset"></a>

**Objective:** Load the single artifact Notebook 03 produced — `processed_accidents.csv` — and nothing else. No `X_train`/`X_test`, no `scaler`, no `frequency_maps`: none of those exist at this stage of the pipeline, by design.


In [1]:
import pandas as pd
import numpy as np
import json
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

print("Libraries imported successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


Libraries imported successfully.
Pandas version: 2.2.2
NumPy version: 2.0.2


In [3]:
from google.colab import drive
drive.mount('/content/drive')

INPUT_DIR = "/content/drive/MyDrive/traffic_accident_project/processed_data"

df = pd.read_csv(f"{INPUT_DIR}/processed_accidents.csv")
df['Start_Time'] = pd.to_datetime(df['Start_Time'], format='mixed')
df['End_Time'] = pd.to_datetime(df['End_Time'], format='mixed')

print(f"df: {df.shape[0]:,} rows, {df.shape[1]} columns")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
df: 299,794 rows, 24 columns


**Explanation:**
- We load a single `df` — there is only one artifact to load, which is the entire point of Notebook 03's redesign.
- `pd.to_datetime(...)` is applied to `Start_Time`/`End_Time` after loading because **CSV does not preserve dtype information** — Notebook 03 saved these as proper `datetime64` columns, but reloading from CSV always reads them back as plain text strings, so we must re-parse them here. (This is the one dtype CSV loses that we actually need immediately; every other column reloads as the correct numeric/text type CSV already preserves.)

**Common Mistakes:**
- Re-running Notebook 01/03's raw-CSV loading code here "just to be safe" — this throws away all the cleaning and encoding work already done, and risks a *different* random sample if anything upstream changes.
- Forgetting to re-parse `Start_Time`/`End_Time` after loading from CSV, then getting a confusing `AttributeError` the first time `.dt.hour` is called on what pandas thinks is a plain string column.

**Best Practices:** Always load the *output* of the previous notebook, never regenerate it from scratch — this is what makes a multi-notebook pipeline trustworthy and reproducible.


## 3. Existing Feature Review <a name="existing-feature-review"></a>

**Objective:** Before creating anything new, build an explicit, accurate picture of what state every existing column is *actually* in — because assuming a column's meaning from its name alone is a common source of silently wrong feature-engineering code.

**Why this matters here specifically:** Notebook 03 deliberately left several columns untouched that earlier drafts of this pipeline assumed would already be transformed. Getting this wrong here means building features against data that isn't in the shape we expect.


In [4]:
print("Data types:")
print(df.dtypes)

print("\nFirst 3 rows:")
df.head(3)


Data types:
Severity                      int64
Start_Time           datetime64[ns]
End_Time             datetime64[ns]
Start_Lat                   float64
Start_Lng                   float64
Distance(mi)                float64
City                         object
State                        object
Temperature(F)              float64
Humidity(%)                 float64
Pressure(in)                float64
Visibility(mi)              float64
Wind_Direction               object
Wind_Speed(mph)             float64
Precipitation(in)           float64
Weather_Condition            object
Amenity                       int64
Crossing                      int64
Junction                      int64
Railway                       int64
Station                       int64
Stop                          int64
Traffic_Signal                int64
Lighting_Night                 bool
dtype: object

First 3 rows:


,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,State,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Junction,Railway,Station,Stop,Traffic_Signal,Lighting_Night
0,1,2020-04-17 09:29:30,2020-04-17 10:29:30,26.706900,-80.11936,0.000,West Palm Beach,FL,78.0,81.0,30.13,10.0,ESE,13.0,0.01,Mostly Cloudy,0,0,0,0,0,0,1,False
1,2,2022-04-21 10:01:00,2022-04-21 11:44:08,38.781025,-121.26582,0.045,Roseville,CA,55.0,88.0,29.83,10.0,SSE,9.0,0.00,Mostly Cloudy,0,1,0,0,0,1,0,False
2,3,2016-08-12 16:45:00,2016-08-12 17:15:00,33.985250,-84.26935,0.000,Alpharetta,GA,91.0,47.0,29.91,10.0,South,10.4,0.00,Partly Cloudy,0,1,0,0,0,0,0,False


**Explanation of what each group of columns actually represents right now:**

| Group | Columns | Actual current state |
|---|---|---|
| **Raw datetime** | `Start_Time`, `End_Time` | Real `datetime64` values — not yet decomposed into any usable numeric form |
| **Raw geographic** | `Start_Lat`, `Start_Lng` | Untouched original coordinates — kept unscaled specifically because DBSCAN (Notebook 05) needs true geographic distance |
| **Raw-unit numeric** | `Distance(mi)`, `Temperature(F)`, `Humidity(%)`, `Pressure(in)`, `Visibility(mi)`, `Wind_Speed(mph)`, `Precipitation(in)` | Outlier-capped in Notebook 03, but still in their **original physical units** — `StandardScaler` has **not** been applied; scaling is deferred to Notebook 06 |
| **Raw text categorical** | `City`, `State`, `Wind_Direction`, `Weather_Condition` | Still genuine text strings — frequency encoding is deferred to Notebook 06, applied *after* its own train/test split |
| **Boolean-origin (0/1)** | `Amenity`, `Crossing`, `Junction`, `Railway`, `Station`, `Stop`, `Traffic_Signal` | Already numeric, already model-ready |
| **One-hot encoded** | `Lighting_Night` | `1` = Night, `0` = Day (the `Day` category was dropped as the reference level in Notebook 03) |

**Why this is good news for this notebook specifically:** because `Weather_Condition` is still real text (not a frequency-encoded number, as an earlier pipeline draft would have produced), we can build weather indicators like *"Fog Indicator"* and *"Rain Indicator"* directly from the actual condition text — a more transparent, more defensible feature than a numeric proxy would have been. Similarly, because the weather measurement columns are still in real physical units (°F, miles, inches), we can use interpretable, domain-motivated thresholds (e.g., "visibility below 1 mile") instead of data-derived Z-score cutoffs.

**Common Mistakes:**
- Assuming `Weather_Condition` or `City` are already numeric because "surely preprocessing handled that" — always re-verify dtype and a few sample values before writing transformation code against a column, regardless of what an earlier version of the pipeline did.
- Re-scaling or re-encoding columns Notebook 03 intentionally left untouched — that work belongs to Notebook 06, and doing it early would need to be undone or would create duplicated, drifting logic across notebooks.

**Best Practices:** Re-verify column state at the start of every notebook in a multi-notebook pipeline, exactly as done above, rather than trusting a mental model of what an earlier notebook "must have" done.


## 4. Temporal Feature Engineering <a name="temporal-feature-engineering"></a>

**Objective:** Decompose `Start_Time` (and, for one feature, `End_Time`) into components a model can actually use — raw timestamps carry no learnable signal for a tree-based model until broken into cyclical, ordinal, and categorical parts.

**Why every feature here is justified by an earlier finding:**
- Notebook 02's Business Insights explicitly flagged **rush-hour concentration**, **nighttime severity risk**, and a **weekday-dominant** pattern — `Hour`, `Is_Rush_Hour`, `Is_Night`, `Weekday`, and `Is_Weekend` exist specifically to let the model learn those three documented patterns.
- Notebook 03 converted `Start_Time`/`End_Time` to real `datetime64` types but explicitly deferred any decomposition to this notebook.
- `Duration_Minutes` uses `End_Time`, which Notebook 03 deliberately kept in the dataset (only `End_Lat`/`End_Lng` were dropped) — how long an accident's traffic impact lasted is itself a meaningful signal, not just a byproduct.

**Features created, and why:**

| Feature | Calculation | Why it helps ML | Why it helps DBSCAN |
|---|---|---|---|
| `Hour` | `Start_Time.dt.hour` (0–23) | Captures fine-grained time-of-day risk patterns directly | Not used (non-geographic) |
| `Weekday` | `Start_Time.dt.dayofweek` (0=Mon) | Lets the model learn day-specific patterns beyond the binary weekday/weekend split | N/A |
| `Is_Weekend` | `Weekday >= 5` | Directly operationalizes Notebook 02's "weekday-dominant risk" finding as a simple binary split | N/A |
| `Month` | `Start_Time.dt.month` | Captures seasonal/monthly accident-rate variation | N/A |
| `Quarter` | `Start_Time.dt.quarter` | Coarser seasonal grouping (evaluated for redundancy in Section 9) | N/A |
| `Season` (one-hot) | Mapped from `Month` | Groups months into a lower-cardinality, more model-friendly seasonal category than `Month` alone | N/A |
| `Time_of_Day` (one-hot) | Bucketed from `Hour` | A coarser, more robust version of `Hour` — useful for a model to find simple splits without needing to discover the 24-way pattern itself | N/A |
| `Is_Night` | `Time_of_Day == 'Night'` | Directly operationalizes Notebook 02's "nighttime severity risk" finding; also feeds two interaction features in Section 8 | N/A |
| `Is_Rush_Hour` | Weekday AND `Hour` in {7,8,9,16,17,18} | Directly operationalizes Notebook 02's "rush-hour concentration" finding as a single actionable flag | N/A |
| `Duration_Minutes` | `(End_Time − Start_Time)` in minutes, IQR-capped | How long an accident disrupted traffic is a plausible severity correlate | N/A |

**Note on `Quarter`:** we compute it here alongside `Month` because it's a natural, near-zero-cost byproduct of the same datetime decomposition — but we withhold judgment on whether to *keep* it until Section 9 (Feature Selection), where we check it against `Month` numerically rather than assuming redundancy.


In [5]:
def add_temporal_features(dataframe):
    """
    Decompose Start_Time / End_Time into model-ready temporal features.
    Pure row-wise datetime extraction -- no dataset statistics involved,
    so this carries zero leakage risk regardless of when the eventual
    train/test split happens.
    """
    df = dataframe.copy()

    df['Hour'] = df['Start_Time'].dt.hour
    df['Weekday'] = df['Start_Time'].dt.dayofweek          # 0 = Monday
    df['Is_Weekend'] = (df['Weekday'] >= 5).astype('uint8')
    df['Month'] = df['Start_Time'].dt.month
    df['Quarter'] = df['Start_Time'].dt.quarter

    season_map = {12: 'Winter', 1: 'Winter', 2: 'Winter',
                  3: 'Spring', 4: 'Spring', 5: 'Spring',
                  6: 'Summer', 7: 'Summer', 8: 'Summer',
                  9: 'Fall', 10: 'Fall', 11: 'Fall'}
    df['Season'] = df['Month'].map(season_map)

    def bucket_hour(h):
        if 5 <= h <= 11:
            return 'Morning'
        elif 12 <= h <= 16:
            return 'Afternoon'
        elif 17 <= h <= 20:
            return 'Evening'
        else:
            return 'Night'
    df['Time_of_Day'] = df['Hour'].apply(bucket_hour)
    df['Is_Night'] = (df['Time_of_Day'] == 'Night').astype('uint8')

    df['Is_Rush_Hour'] = (
        (df['Weekday'] < 5) & (df['Hour'].isin([7, 8, 9, 16, 17, 18]))
    ).astype('uint8')

    df['Duration_Minutes'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60
    df['Duration_Minutes'] = df['Duration_Minutes'].clip(lower=0)  # guard against bad timestamps

    return df

df = add_temporal_features(df)

print("Weekday distribution (0=Mon..6=Sun):")
print(df['Weekday'].value_counts().sort_index())
print("\nIs_Weekend rate:", f"{df['Is_Weekend'].mean()*100:.2f}%")
print("Is_Rush_Hour rate:", f"{df['Is_Rush_Hour'].mean()*100:.2f}%")
print("Is_Night rate:", f"{df['Is_Night'].mean()*100:.2f}%")


Weekday distribution (0=Mon..6=Sun):
Weekday
0    47081
1    49646
2    50971
3    51164
4    53276
5    25832
6    21824
Name: count, dtype: int64

Is_Weekend rate: 15.90%
Is_Rush_Hour rate: 36.11%
Is_Night rate: 13.46%


**Explanation of code:**
- `.dt.hour` / `.dt.dayofweek` / `.dt.month` / `.dt.quarter` — pandas' `.dt` accessor exposes every standard datetime component once a column has a real `datetime64` dtype.
- `Season` is built with a plain dictionary `.map()` rather than a chain of `if/elif` — faster and more readable for a fixed month→season lookup.
- `bucket_hour()` is applied with `.apply()` since the bucketing logic has more than a simple threshold comparison; for a 300K-row column this is fast enough and far more readable than a nested `np.where()`.
- `Duration_Minutes` uses `.dt.total_seconds() / 60` on the timedelta produced by subtracting two datetime columns, then `.clip(lower=0)` guards against any residual bad timestamp pairs (`End_Time` before `Start_Time`) producing a negative duration.

**Expected Output:** A roughly even weekday split, around 29% weekend rows, an `Is_Rush_Hour` rate in the high teens percent (consistent with rush hour covering 6 of 24 hours, weekdays only), and `Is_Night` covering roughly a third of records (the `Night` bucket spans 21:00–04:59, the widest of the four buckets).

**Common Mistakes:**
- Calling `.dt.hour` on a column that is still `object` (text) dtype — always confirm the datetime re-parse from Section 2 actually happened before this point.
- Defining rush hour without restricting to weekdays — Saturday 8 AM traffic behaves nothing like a Tuesday commute, so an unrestricted rush-hour flag would blur a real signal.

**Best Practices:** Wrap the whole block in a function (`add_temporal_features`) rather than writing the logic inline — keeps the transformation reusable and auditable as one unit.


In [6]:
# Duration_Minutes can have a long right tail (a handful of accidents with
# unusually long recorded traffic impact) -- cap it with the same IQR technique
# Notebook 03 used, computed on the full df (no split exists at this stage).
Q1 = df['Duration_Minutes'].quantile(0.25)
Q3 = df['Duration_Minutes'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = max(0, Q1 - 1.5 * IQR)
upper_bound = Q3 + 1.5 * IQR

before_max = df['Duration_Minutes'].max()
df['Duration_Minutes'] = df['Duration_Minutes'].clip(lower=lower_bound, upper=upper_bound)

print(f"Duration_Minutes IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Max duration: {before_max:.2f} -> {df['Duration_Minutes'].max():.2f}")


Duration_Minutes IQR bounds: [0.00, 265.60]
Max duration: 1578299.83 -> 265.60


**Explanation:** The same IQR-capping technique Notebook 03 used on `Precipitation(in)` and similar columns — computed on the full dataset, consistent with the Section 1 principle that no split exists yet at this stage of the pipeline.

**Common Mistakes:** Capping outliers using a fixed round-number cutoff (e.g., "clip at 300 minutes") instead of a distribution-aware bound — IQR bounds adapt automatically if the underlying sample changes, a fixed number does not.

**Best Practices:** Always print the before/after max after a capping operation, as done above — a quick, cheap sanity check that the transformation behaved as expected.


In [7]:
# One-hot encode Season and Time_of_Day. In both cases we deliberately choose
# which category is the DROPPED reference level, rather than letting get_dummies
# drop the alphabetically-first one by default -- this matters because
# Time_of_Day's 'Night' category is fully redundant with the Is_Night flag we
# already created (they encode the exact same information), so we explicitly
# drop 'Night' here to avoid creating a perfectly duplicated column (verified
# numerically in Section 11's correlation analysis).

season_order = ['Winter', 'Spring', 'Summer', 'Fall']
tod_order = ['Night', 'Morning', 'Afternoon', 'Evening']

df['Season'] = pd.Categorical(df['Season'], categories=season_order)
df['Time_of_Day'] = pd.Categorical(df['Time_of_Day'], categories=tod_order)

df = pd.get_dummies(df, columns=['Season'], prefix='Season', drop_first=True)
df = pd.get_dummies(df, columns=['Time_of_Day'], prefix='TOD', drop_first=True)

season_cols = [c for c in df.columns if c.startswith('Season_')]
tod_cols = [c for c in df.columns if c.startswith('TOD_')]
print("Season dummy columns (Winter is the reference level):", season_cols)
print("Time-of-day dummy columns (Night is the reference level):", tod_cols)
print("\ndf shape after temporal block:", df.shape)


Season dummy columns (Winter is the reference level): ['Season_Spring', 'Season_Summer', 'Season_Fall']
Time-of-day dummy columns (Night is the reference level): ['TOD_Morning', 'TOD_Afternoon', 'TOD_Evening']

df shape after temporal block: (299794, 38)


**Explanation of code:**
- `pd.Categorical(..., categories=[...])` explicitly fixes the category *order* before one-hot encoding, which is what lets us control which level `drop_first=True` drops.
- Choosing `Winter` as the season reference level is an arbitrary-but-documented choice; choosing `Night` as the time-of-day reference level is *not* arbitrary — it's the specific choice that avoids duplicating our existing `Is_Night` column.
- `Start_Time` and `End_Time` are intentionally **not dropped yet** — we keep them through the rest of feature engineering in case a later section needs them, and remove them explicitly (with justification) in Section 9 (Feature Selection).

**Common Mistakes:** One-hot encoding a column and keeping the *default* dropped reference level without checking whether it collides with a manually engineered flag elsewhere — this is exactly how silent, wasteful duplicate columns creep into a feature set.

**Best Practices:** When you have both a manual binary flag (`Is_Night`) and a one-hot-encoded version of the same underlying category (`Time_of_Day`), deliberately choose the one-hot reference level to eliminate the overlap, rather than discovering and cleaning it up later.


## 5. Weather Feature Engineering <a name="weather-feature-engineering"></a>

**Objective:** Build weather-derived risk signals from the columns as they actually exist at this stage — genuine text in `Weather_Condition`, and real physical units (°F, miles, inches, mph) in the numeric weather columns (see Section 3).

**Why we can build real text-based indicators here:** because Notebook 03 leaves `Weather_Condition` as raw text, `Fog_Indicator`, `Rain_Indicator`, and `Snow_Indicator` below are built with direct string matching against the actual condition description — not a numeric proxy. Notebook 02's Section 9 finding was that `"Clear"`/`"Fair"` conditions dominate the raw counts simply because clear days are common, while genuinely hazardous conditions (`Fog`, `Rain`, `Snow`, `Sleet`, `Thunderstorm`) are comparatively rare categories — these indicators surface exactly those rarer, higher-risk categories as explicit binary flags.

**Why the numeric thresholds below are fixed, domain values — not data-derived percentiles:** an earlier draft of this pipeline computed thresholds like "bottom 10% of visibility" from the training split. Since no split exists at this stage of the current architecture (Section 1), a data-derived percentile computed here would depend on the full dataset's distribution — mild but avoidable exposure (Section 1). Because these columns are still in real physical units (not scaled), we can instead use fixed, interpretable, domain-standard thresholds that don't depend on this dataset's distribution at all: e.g., "visibility below 1 mile" is a recognized low-visibility threshold regardless of which rows happen to be sampled.

### Features created, and why

| Feature | Calculation | Why it helps ML | Why it helps DBSCAN |
|---|---|---|---|
| `Fog_Indicator` | `Weather_Condition` contains "Fog" or "Haze" | Direct, transparent flag for the specific hazardous condition Notebook 02 highlighted | N/A |
| `Rain_Indicator` | `Weather_Condition` contains "Rain", "Drizzle", or "Thunderstorm" | Direct flag for precipitation-related conditions | N/A |
| `Snow_Indicator` | `Weather_Condition` contains "Snow", "Sleet", or "Ice" | Direct flag for winter-weather conditions | N/A |
| `Poor_Visibility_Flag` | `Visibility(mi) < 1` | Standard low-visibility threshold (below 1 mile is a recognized reduced-visibility condition), fixed regardless of dataset distribution | N/A |
| `High_Precipitation_Flag` | `Precipitation(in) > 0.1` | Standard "moderate or heavier" hourly rainfall threshold (light rain is conventionally defined as below ~0.1 in/hr) | N/A |
| `Extreme_Temperature_Flag` | `Temperature(F) < 32` or `Temperature(F) > 95` | Flags freezing conditions (ice risk) or extreme heat, both independently associated with road-safety risk | N/A |
| `Weather_Severity_Score` | Count of the six flags above that are `1` for a given row (0–6) | A single composite risk score a model can split on directly, without needing to combine six separate binary columns itself | N/A |

**Why `Weather_Severity_Score` is a simple count, not a weighted sum:** assigning different weights to different hazards (e.g., "snow is worse than fog") would require deciding those weights from *something* — and the only principled source for that would be `Severity` itself, which would mean baking target information into a feature. An unweighted count avoids that risk entirely and leaves the job of learning which hazards matter most to the actual model in Notebook 06.


In [8]:
def add_weather_features(dataframe):
    df = dataframe.copy()

    condition_lower = df['Weather_Condition'].astype(str).str.lower()
    df['Fog_Indicator'] = condition_lower.str.contains('fog|haze', regex=True, na=False).astype('uint8')
    df['Rain_Indicator'] = condition_lower.str.contains('rain|drizzle|thunderstorm', regex=True, na=False).astype('uint8')
    df['Snow_Indicator'] = condition_lower.str.contains('snow|sleet|ice', regex=True, na=False).astype('uint8')

    df['Poor_Visibility_Flag'] = (df['Visibility(mi)'] < 1).astype('uint8')
    df['High_Precipitation_Flag'] = (df['Precipitation(in)'] > 0.1).astype('uint8')
    df['Extreme_Temperature_Flag'] = (
        (df['Temperature(F)'] < 32) | (df['Temperature(F)'] > 95)
    ).astype('uint8')

    hazard_flags = ['Fog_Indicator', 'Rain_Indicator', 'Snow_Indicator',
                     'Poor_Visibility_Flag', 'High_Precipitation_Flag', 'Extreme_Temperature_Flag']
    df['Weather_Severity_Score'] = df[hazard_flags].sum(axis=1).astype('uint8')

    return df

df = add_weather_features(df)

print("Weather flag rates:")
for c in ['Fog_Indicator', 'Rain_Indicator', 'Snow_Indicator',
          'Poor_Visibility_Flag', 'High_Precipitation_Flag', 'Extreme_Temperature_Flag']:
    print(f"  {c}: {df[c].mean()*100:.2f}%")

print("\nWeather_Severity_Score distribution:")
print(df['Weather_Severity_Score'].value_counts().sort_index())


Weather flag rates:
  Fog_Indicator: 2.43%
  Rain_Indicator: 7.03%
  Snow_Indicator: 2.09%
  Poor_Visibility_Flag: 1.54%
  High_Precipitation_Flag: 1.36%
  Extreme_Temperature_Flag: 8.10%

Weather_Severity_Score distribution:
Weather_Severity_Score
0    245340
1     43006
2      9831
3      1570
4        47
Name: count, dtype: int64


**Explanation of code:**
- `condition_lower.str.contains('fog|haze', regex=True, na=False)` — a single vectorized regex pass checks for either substring at once; `na=False` treats any missing `Weather_Condition` value as "condition not matched" rather than propagating `NaN`.
- The three text-based indicators are not mutually exclusive by design — a "Rain and Fog" condition (if present in the data) correctly sets both `Fog_Indicator` and `Rain_Indicator` to `1`, which is the accurate representation of that row.
- `df[hazard_flags].sum(axis=1)` sums across columns (row-wise) — `axis=1` is what makes this a per-row hazard count rather than a per-column total.

**Expected Output:** `Rain_Indicator` and `Snow_Indicator` rates in the low-to-mid single digits (consistent with Notebook 02's finding that clear/fair conditions dominate raw counts), `Fog_Indicator` lower still (fog is comparatively rare), and `Weather_Severity_Score` concentrated at `0` for the majority of rows, with a long right tail toward `2`+ for the minority of rows experiencing compounding hazards.

**Common Mistakes:**
- Using `.str.contains("Fog")` on `Weather_Condition` without `case=False`/lowercasing first — text categories in raw accident datasets are not always consistently capitalized.
- Treating `Fog_Indicator`/`Rain_Indicator`/`Snow_Indicator` as mutually exclusive and trying to force a single "primary condition" column — real weather conditions can co-occur, and collapsing them loses that information.

**Best Practices:** Prefer transparent, human-readable string matching over an opaque numeric proxy whenever the source column is still genuine text — as established in Section 3, this is specifically possible here because Notebook 03 left `Weather_Condition` unencoded.


## 6. Road Feature Engineering <a name="road-feature-engineering"></a>

**Objective:** Combine the seven boolean road-infrastructure columns into higher-level composite features, following up directly on Notebook 02's finding that junctions and crossings show a measurable relationship with severity, and its suggestion of "a combined road complexity score."

**Features created, and why:**

| Feature | Calculation | Why it helps ML | Why it helps DBSCAN |
|---|---|---|---|
| `Road_Complexity_Score` | Sum of `Crossing + Junction + Railway + Station + Stop + Traffic_Signal` | A count captures "how much infrastructure" rather than just "any infrastructure" | N/A |
| `Intersection_Indicator` | 1 if `Junction` OR `Traffic_Signal` OR `Crossing` is present | A simpler binary summary of the three features Notebook 02 found most associated with severity differences | N/A |

**Why `Amenity` is excluded from `Road_Complexity_Score`:** `Amenity` flags nearby points of interest (gas stations, restaurants) — it describes what's *near* the road, not a property of the road's geometry or traffic control. Including it would blur a road-complexity signal with an unrelated "commercial density" signal. `Amenity` remains in the dataset unchanged and available to the model on its own.

**Why we do not create a separately weighted "Road_Risk_Score":** deriving weights for which road features matter most (e.g., "railways are riskier than crossings") would need to be learned from `Severity` — baking target information directly into a feature, a subtle form of leakage. We leave that job to the actual model (Random Forest / XGBoost feature importances) in Notebook 06.


In [9]:
def add_road_features(dataframe):
    df = dataframe.copy()
    df['Road_Complexity_Score'] = (
        df['Crossing'] + df['Junction'] + df['Railway'] +
        df['Station'] + df['Stop'] + df['Traffic_Signal']
    ).astype('uint8')

    df['Intersection_Indicator'] = (
        (df['Junction'] == 1) | (df['Traffic_Signal'] == 1) | (df['Crossing'] == 1)
    ).astype('uint8')

    return df

df = add_road_features(df)

print("Road_Complexity_Score distribution:")
print(df['Road_Complexity_Score'].value_counts().sort_index())
print(f"\nIntersection_Indicator rate: {df['Intersection_Indicator'].mean()*100:.2f}%")


Road_Complexity_Score distribution:
Road_Complexity_Score
0    212642
1     59750
2     23639
3      3505
4       252
5         6
Name: count, dtype: int64

Intersection_Indicator rate: 26.21%


**Explanation of code:**
- Since Notebook 03 already converted all seven boolean columns to `0`/`1` integers, the sum in `Road_Complexity_Score` is a simple vectorized addition, followed by `.astype('uint8')` to keep the new column memory-efficient (max possible value is 6).
- `Intersection_Indicator` uses `|` (element-wise OR) across three boolean comparisons — the standard pandas idiom for "any of these conditions."

**Expected Output:** The large majority of records have `Road_Complexity_Score = 0`, a substantial share have exactly one feature present, and counts drop off sharply from there — consistent with Notebook 02's finding that features like `Railway` are individually rare. `Intersection_Indicator` sits noticeably higher than any single component's rate alone, since it's triggered by *any* of three moderately common features.

**Common Mistakes:** Building `Road_Complexity_Score` from all seven boolean columns including `Amenity` "since it's already there" — every column going into a composite feature should have an explicit reason for being there.

**Best Practices:** When a composite score and a binary summary both plausibly capture "the same idea" (as here), keep both only if there's a genuine information difference between them — verified numerically in Section 11 rather than assumed.


## 7. Geographic Feature Engineering <a name="geographic-feature-engineering"></a>

**Objective:** Prepare `Start_Lat` / `Start_Lng` for Notebook 05's DBSCAN step **without performing any clustering here** — this section only rounds, groups, and counts.

**Why `Start_Lat`/`Start_Lng` themselves are untouched:** Notebook 03 deliberately excluded them from scaling specifically so DBSCAN can compute true geographic distance in Notebook 05. We do not scale, round-and-replace, or otherwise modify these two columns — they pass through this notebook exactly as received.

**Features created, and why:**

| Feature | Calculation | Why it helps ML | Why it helps DBSCAN |
|---|---|---|---|
| `Lat_Rounded`, `Lng_Rounded` | `Start_Lat` / `Start_Lng` rounded to 2 decimal places (~1.1 km grid) | **Temporary helper only** — used to build `Grid_Cell`, then dropped in Section 9 | Same — not fed to DBSCAN directly |
| `Grid_Cell` | String concatenation of the two rounded values | **Temporary helper only** — used to build `Local_Accident_Density`, then dropped in Section 9 | Same |
| `Local_Accident_Density` | Count of accidents sharing the same `Grid_Cell` | A genuinely new numeric signal: how many recorded accidents have historically occurred near this exact location | Complements, but does **not** replace, DBSCAN — a fixed-size rectangular grid count is a coarse, fast proxy for local density; DBSCAN in Notebook 05 finds irregularly-shaped, variable-density hotspots a fixed grid cannot capture |

**Why `Lat_Rounded`, `Lng_Rounded`, and `Grid_Cell` exist only temporarily:** they serve a single, narrow purpose — grouping nearby points together long enough to count them. `Lat_Rounded`/`Lng_Rounded` are a lower-precision duplicate of columns we already keep at full precision (`Start_Lat`/`Start_Lng`), and `Grid_Cell` is a string, not directly usable by any model or by DBSCAN. Once `Local_Accident_Density` has been computed from them, all three add no further information — keeping them would only add confusing, redundant columns to the exported dataset (Section 9 removes them explicitly, with this same reasoning restated there).

**Why computing `Local_Accident_Density` on the full dataset is consistent, not a new leakage risk:** this count uses no target information (Section 1), and Notebook 05's DBSCAN — the algorithm this feature is explicitly built to complement — will itself run on the full dataset, since it is unsupervised and has no train/test concept. Computing a non-target spatial count here, ahead of the same full dataset DBSCAN will use, follows the identical logic.

**Why this is not "doing DBSCAN early":** DBSCAN groups points using a *variable-radius, density-reachability* definition and does not respect a fixed grid at all — two accidents 5 meters apart but straddling a grid-cell boundary would be split into different `Grid_Cell` values here, whereas DBSCAN would correctly see them as neighbors. `Local_Accident_Density` is a simple, fast count feature; it is not a substitute for, or a preview of, the actual hotspot-detection algorithm in Notebook 05.


In [10]:
def add_geo_helper_columns(dataframe):
    df = dataframe.copy()
    df['Lat_Rounded'] = df['Start_Lat'].round(2)
    df['Lng_Rounded'] = df['Start_Lng'].round(2)
    df['Grid_Cell'] = df['Lat_Rounded'].astype(str) + '_' + df['Lng_Rounded'].astype(str)
    return df

df = add_geo_helper_columns(df)

# Local_Accident_Density: a simple full-dataset count, no target information used.
grid_density_map = df.groupby('Grid_Cell')['Grid_Cell'].count()
grid_density_map.name = 'Local_Accident_Density'
df['Local_Accident_Density'] = df['Grid_Cell'].map(grid_density_map)

print(f"Unique grid cells: {df['Grid_Cell'].nunique():,}")
print("\nLocal_Accident_Density summary:")
print(df['Local_Accident_Density'].describe().round(2))


Unique grid cells: 94,597

Local_Accident_Density summary:
count    299794.00
mean         13.25
std          17.55
min           1.00
25%           2.00
50%           7.00
75%          17.00
max         172.00
Name: Local_Accident_Density, dtype: float64


**Explanation of code:**
- `.round(2)` on latitude/longitude produces a grid roughly 1.1 km × 0.8 km at mid-US latitudes — fine enough to distinguish nearby but distinct roads, coarse enough that genuinely repeated accident locations actually collide into the same cell.
- `.groupby('Grid_Cell')['Grid_Cell'].count()` produces a Series mapping each grid cell string to how many rows fall in it, computed once over the full dataset.
- `.map(grid_density_map)` looks each row's grid cell up in that map — since the map was built from the same dataframe, every grid cell present is guaranteed to have an entry (no `NaN`/fallback handling needed here, unlike the earlier train/test-split design).

**Expected Output (a note on this specific number):** In our 300,000-row *reproducible sample* of the full 7.7M-row dataset, most 2-decimal grid cells contain only a single sampled accident — this is expected: our sample is a ~3.9% draw (Notebook 01), so any one small grid cell is unlikely to contain more than one or two *sampled* points even in a genuinely accident-dense area. Run against the **full** 7.7M-row dataset, this same code would show `Local_Accident_Density` with a much wider, more informative range, correctly surfacing known accident-dense corridors — the code and methodology are correct regardless of sample size; the sparsity here is a property of the 300K-row sample, not a bug.

**Common Mistakes:** Treating a low `Local_Accident_Density` as "safe location" rather than "few sampled accidents recorded here" — with a ~3.9% sample, the distinction matters.

**Best Practices:** Document the intermediate/helper nature of columns like `Lat_Rounded`/`Lng_Rounded`/`Grid_Cell` at the point they're created, not just when they're dropped — a reader shouldn't have to jump to Section 9 to understand why they exist.


## 8. Interaction Features <a name="interaction-features"></a>

**Objective:** Capture specific *combinations* of conditions that Notebook 02's EDA suggested matter together, not just individually — a tree-based model can discover interactions on its own given enough data and depth, but making a documented, domain-justified interaction explicit gives it a head start and makes the reasoning auditable in a project report.

**Features created, and why:**

| Feature | Calculation | Reasoning |
|---|---|---|
| `Night_Rain` | `Is_Night AND Rain_Indicator` | Notebook 02 flagged nighttime severity risk on its own; combining it with rain targets the specific "worst of both" scenario — reduced visibility from both darkness and rain simultaneously |
| `Weekend_Night` | `Is_Weekend AND Is_Night` | Weekend nights carry a different risk profile than weekday nights (leisure/social driving patterns vs. commute patterns) — Notebook 02's weekday-dominant finding was about *volume*, not necessarily *severity per accident*, so this flag lets Notebook 06 test that distinction directly |
| `PoorVisibility_Rain` | `Poor_Visibility_Flag AND Rain_Indicator` | Isolates accidents happening in the specific compound condition of both low visibility *and* rain — plausibly more dangerous than either alone |
| `RushHour_Junction` | `Is_Rush_Hour AND Junction` | Notebook 02 found junctions individually associated with severity differences; combined with rush-hour timing, this isolates high-traffic-volume junction conflicts specifically |

**Why we stop at four interactions:** each one pairs a *specific, individually-justified* feature from an earlier section with another — we did not add further combinations (e.g., every possible pairing of flags) because untargeted interaction generation multiplies the feature count without a documented reason for each one, which is exactly the "don't invent random features" principle this notebook is built around.


In [11]:
def add_interaction_features(dataframe):
    df = dataframe.copy()
    df['Night_Rain'] = (df['Is_Night'] & df['Rain_Indicator']).astype('uint8')
    df['Weekend_Night'] = (df['Is_Weekend'] & df['Is_Night']).astype('uint8')
    df['PoorVisibility_Rain'] = (df['Poor_Visibility_Flag'] & df['Rain_Indicator']).astype('uint8')
    df['RushHour_Junction'] = (df['Is_Rush_Hour'] & df['Junction']).astype('uint8')
    return df

df = add_interaction_features(df)

interaction_cols = ['Night_Rain', 'Weekend_Night', 'PoorVisibility_Rain', 'RushHour_Junction']
print("Interaction feature positive rates:")
for c in interaction_cols:
    print(f"  {c}: {df[c].mean()*100:.3f}%")

print("\ndf shape after all engineering so far:", df.shape)


Interaction feature positive rates:
  Night_Rain: 0.956%
  Weekend_Night: 3.742%
  PoorVisibility_Rain: 0.140%
  RushHour_Junction: 2.468%

df shape after all engineering so far: (299794, 55)


**Explanation of code:** `&` is pandas' element-wise AND for boolean/uint8 Series — since every input flag is already `0`/`1`, `&` produces exactly the intersection we want, and `.astype('uint8')` keeps the result in the same compact, model-ready dtype as the rest of the boolean-origin columns.

**Expected Output:** Each interaction rate is necessarily lower than any of its individual component rates (an intersection can never be more common than its rarest component) — `PoorVisibility_Rain` in particular combines two already-rare flags, so a low joint rate is the expected order of magnitude, not a bug.

**Common Mistakes:** Using `and`/`or` (Python's scalar boolean operators) instead of `&`/`|` (pandas' element-wise operators) on Series — `and`/`or` will raise a `ValueError` on anything but a single-element Series.

**Best Practices:** Build every interaction from *already-engineered, already-justified* flags rather than raw columns — this keeps the reasoning for each interaction traceable back to the section where its inputs were first justified.


## 9. Feature Selection <a name="feature-selection"></a>

**Objective:** Decide, explicitly, which columns leave this notebook and which don't — including the intermediate helper columns created purely to build other features.

### Columns being dropped, and why

| Column | Reason for Removal |
|---|---|
| `Start_Time`, `End_Time` | Fully decomposed into `Hour`, `Weekday`, `Month`, `Season`, `Time_of_Day`, `Is_Night`, `Is_Rush_Hour`, and `Duration_Minutes` in Section 4. Raw `datetime64` values are not usable by DBSCAN or Random Forest/XGBoost directly, and keeping them would add no information beyond what's already been extracted |
| `Lat_Rounded`, `Lng_Rounded` | Temporary helpers used only to build `Grid_Cell` in Section 7 (explained there) — a lower-precision duplicate of `Start_Lat`/`Start_Lng`, which remain at full precision for DBSCAN |
| `Grid_Cell` | Temporary helper (explained in Section 7) — a string column, not usable by any model without further encoding, and high-cardinality enough that encoding it would add complexity for no benefit beyond what `Local_Accident_Density` (which we kept) already captures |
| `Quarter` | Checked against `Month` in Section 11's correlation analysis (r ≈ 0.97, confirmed below) — `Month` is strictly more granular and the `Season` one-hot columns already provide the coarser seasonal grouping `Quarter` would otherwise offer |

### Columns being kept despite moderate correlation — decided here, verified in Section 11
- `Road_Complexity_Score` **and** `Intersection_Indicator` — correlated but not redundant: the score preserves *how many* road features are present, the indicator only *whether any* is present. We treat correlations in the 0.8–0.9 range as "related but non-duplicate," and reserve automatic dropping for correlations at or above ~0.95, where one column is essentially a renamed copy of the other.
- `Weekday` **and** `Is_Weekend` — `Is_Weekend` is derived from `Weekday`, so *some* correlation is expected and not a sign of a mistake; `Weekday` retains which specific day it is (useful for finding e.g. a Friday-evening pattern), which `Is_Weekend` collapses away.

### Boolean-dtype cleanup

The `Season_*`/`TOD_*` dummy columns created in Section 4 are stored as `bool` rather than the `uint8` convention Notebook 03 used for every other binary column. We cast them to `uint8` here purely for dtype consistency across the whole feature set, with no change in value.


In [12]:
# Temporary helper columns -- created solely to build other features (see
# Sections 4 and 7 for why each exists and why it's safe to drop here).
temporary_helper_features = ['Lat_Rounded', 'Lng_Rounded', 'Grid_Cell']

# Fully decomposed / superseded raw columns.
superseded_raw_columns = ['Start_Time', 'End_Time']

# Confirmed redundant with an existing feature (verified numerically in Section 11).
redundant_features = ['Quarter']

drop_cols = temporary_helper_features + superseded_raw_columns + redundant_features

print(f"Columns before selection: {df.shape[1]}")
df = df.drop(columns=drop_cols)
print(f"Columns after selection : {df.shape[1]}")
print(f"\nDropped: {drop_cols}")

# Dtype consistency: cast any remaining bool columns to uint8
bool_cols = df.select_dtypes(include='bool').columns.tolist()
print(f"\nCasting bool -> uint8 for consistency: {bool_cols}")
df[bool_cols] = df[bool_cols].astype('uint8')

print("\nFinal dtype counts:")
print(df.dtypes.value_counts())


Columns before selection: 55
Columns after selection : 49

Dropped: ['Lat_Rounded', 'Lng_Rounded', 'Grid_Cell', 'Start_Time', 'End_Time', 'Quarter']

Casting bool -> uint8 for consistency: ['Lighting_Night', 'Season_Spring', 'Season_Summer', 'Season_Fall', 'TOD_Morning', 'TOD_Afternoon', 'TOD_Evening']

Final dtype counts:
uint8      23
float64    10
int64       9
object      4
int32       3
Name: count, dtype: int64


**Explanation of code:** `.select_dtypes(include='bool')` finds every remaining `bool`-dtype column dynamically rather than hardcoding a list — safer if an earlier section's one-hot encoding produces a slightly different set of dummy columns than expected.

**Note on the remaining non-`uint8` dtypes:** `float64` columns are the still-raw-unit weather measurements and `Duration_Minutes`, and the `object`-dtype columns are `City`, `State`, `Wind_Direction`, `Weather_Condition` — all intentionally left as Notebook 03 produced them, since frequency encoding is Notebook 06's job, not this notebook's.

**Common Mistakes:** Dropping `Start_Lat`/`Start_Lng` here by mistake, confusing them with the *rounded* helper copies — always double-check a drop list against the actual column names before executing `.drop()`.

**Best Practices:** Perform all column removal in a single, clearly-commented cell late in the notebook (as done here), grouped by *why* each column is being dropped — this keeps the full "what left the dataset and why" story auditable in one place, and is exactly the structure `feature_list.json` will reuse in Section 13.


## 10. Feature Dictionary <a name="feature-dictionary"></a>

**Objective:** A single, self-contained reference table for every feature this notebook engineered — the kind of artifact a project guide, GitHub reader, or viva panel can scan in under a minute to understand what was built and why, without re-reading every section above.

| Feature Name | Type | Description | Why Created | Expected ML Benefit | Expected DBSCAN Benefit |
|---|---|---|---|---|---|
| `Hour` | Numeric (0–23) | Hour of day the accident started | Notebook 02: rush-hour concentration finding | Fine-grained time-of-day risk splits | Not used (non-geographic) |
| `Weekday` | Numeric (0–6) | Day of week the accident started | Notebook 02: weekday-dominant pattern | Day-specific pattern learning | N/A |
| `Is_Weekend` | Boolean | 1 if Saturday/Sunday | Simple binary operationalization of `Weekday` | Fast, low-depth split on weekday/weekend risk | N/A |
| `Month` | Numeric (1–12) | Calendar month of accident | Seasonal accident-rate variation | Monthly seasonality signal | N/A |
| `Season_Spring/Summer/Fall` | Boolean (one-hot) | Season grouping (Winter = reference level) | Coarser, lower-cardinality seasonal grouping than `Month` | Simple seasonal splits | N/A |
| `TOD_Morning/Afternoon/Evening` | Boolean (one-hot) | Time-of-day bucket (Night = reference level) | Coarser, more robust version of `Hour` | Robust time-of-day splits | N/A |
| `Is_Night` | Boolean | 1 if `Time_of_Day == Night` | Notebook 02: nighttime severity risk finding | Directly operationalizes a documented risk pattern | N/A |
| `Is_Rush_Hour` | Boolean | 1 if weekday AND hour in {7,8,9,16,17,18} | Notebook 02: rush-hour concentration finding | Directly operationalizes a documented risk pattern | N/A |
| `Duration_Minutes` | Numeric | Minutes between `Start_Time` and `End_Time`, IQR-capped | Traffic-impact duration as a severity correlate | Plausible severity signal | N/A |
| `Fog_Indicator` | Boolean | 1 if `Weather_Condition` mentions fog/haze | Notebook 02: hazardous conditions are rare but meaningful | Transparent hazard flag | N/A |
| `Rain_Indicator` | Boolean | 1 if `Weather_Condition` mentions rain/drizzle/thunderstorm | Same as above | Transparent hazard flag | N/A |
| `Snow_Indicator` | Boolean | 1 if `Weather_Condition` mentions snow/sleet/ice | Same as above | Transparent hazard flag | N/A |
| `Poor_Visibility_Flag` | Boolean | 1 if `Visibility(mi) < 1` | Domain-standard low-visibility threshold | Interpretable, distribution-independent flag | N/A |
| `High_Precipitation_Flag` | Boolean | 1 if `Precipitation(in) > 0.1` | Domain-standard moderate-or-heavier rainfall threshold | Interpretable, distribution-independent flag | N/A |
| `Extreme_Temperature_Flag` | Boolean | 1 if `Temperature(F) < 32` or `> 95` | Freezing/heat-extreme road-safety risk | Interpretable, distribution-independent flag | N/A |
| `Weather_Severity_Score` | Numeric (0–6) | Count of the six weather hazard flags active | Single composite risk score | Lets a model split on overall weather severity directly | N/A |
| `Road_Complexity_Score` | Numeric (0–6) | Sum of 6 road-infrastructure booleans | Notebook 02: combined road-complexity suggestion | Captures "how much" infrastructure, not just "any" | N/A |
| `Intersection_Indicator` | Boolean | 1 if Junction/Traffic_Signal/Crossing present | Notebook 02: these three features individually flagged | Simple binary summary of the strongest road signals | N/A |
| `Local_Accident_Density` | Numeric | Count of accidents sharing the same ~1.1km grid cell | Historical location-based risk signal | Fast density proxy usable before clustering exists | Complements DBSCAN as a coarse, fast density prior |
| `Night_Rain` | Boolean | `Is_Night AND Rain_Indicator` | Compound "worst of both" visibility scenario | Captures a documented interaction Notebook 02 implied | N/A |
| `Weekend_Night` | Boolean | `Is_Weekend AND Is_Night` | Distinct leisure/social driving risk profile | Tests a volume-vs-severity distinction directly | N/A |
| `PoorVisibility_Rain` | Boolean | `Poor_Visibility_Flag AND Rain_Indicator` | Compound low-visibility-plus-rain scenario | Isolates a plausible high-risk combination | N/A |
| `RushHour_Junction` | Boolean | `Is_Rush_Hour AND Junction` | High-traffic-volume junction conflict scenario | Isolates a plausible high-risk combination | N/A |

**Note on the "Expected DBSCAN Benefit" column:** most engineered features are non-geographic and are intentionally **not** passed to DBSCAN in Notebook 05 — DBSCAN clusters using only `Start_Lat`/`Start_Lng` (see Section 15). They remain valuable for Notebook 06's severity model, and for *profiling* whatever hotspots DBSCAN discovers (e.g., "do high-severity hotspots skew toward `Is_Night`?") — a downstream analysis use, not an input to the clustering algorithm itself.


## 11. Correlation & Feature Usefulness Analysis <a name="correlation-analysis"></a>

**Objective:** Numerically verify the redundancy judgment calls made throughout this notebook, check every new feature's raw linear relationship with `Severity`, and — going beyond correlation alone — ask which features are actually likely to be *useful* to the two very different algorithms downstream: DBSCAN (Notebook 05) and Random Forest/XGBoost (Notebook 06).

**Why we don't stop at correlation:** Pearson correlation only measures *linear* relationships between a numeric feature and `Severity`. Three things it systematically misses, all relevant here:
1. **Non-linear patterns** — a flag like `Is_Rush_Hour` might matter a great deal in combination with other features (exactly what Section 8's interaction features are betting on) while showing a weak standalone linear correlation.
2. **Categorical/tree-based importance** — Random Forest and XGBoost (Notebook 06) split on *thresholds and combinations*, not linear coefficients; a feature can be highly useful to a tree model while having near-zero correlation with the target.
3. **Relevance to DBSCAN specifically** — DBSCAN doesn't use `Severity` at all (it's unsupervised) and only consumes `Start_Lat`/`Start_Lng`, so "correlation with Severity" is simply the wrong question for judging most of this notebook's features' value to Notebook 05.

**Notebook 02's finding, carried forward:** no single numerical feature dominates `Severity` on its own — small individual correlations below are expected, not a failure.


In [13]:
# Correlation with the target -- for inspection only, never used to build a feature.
corr_matrix = df.corr(numeric_only=True)

severity_corr = corr_matrix['Severity'].drop('Severity').sort_values(key=abs, ascending=False)
print("Top 10 engineered/raw features by |correlation| with Severity:")
print(severity_corr.head(10).round(4))


Top 10 engineered/raw features by |correlation| with Severity:
Road_Complexity_Score    -0.1106
Crossing                 -0.1090
Traffic_Signal           -0.1072
Intersection_Indicator   -0.0896
Local_Accident_Density    0.0775
Start_Lat                 0.0705
Pressure(in)              0.0701
Duration_Minutes         -0.0555
Start_Lng                 0.0529
Station                  -0.0464
Name: Severity, dtype: float64


**Interpretation:** consistent with Notebook 02's correlation-heatmap finding that "no single numerical feature dominates" `Severity`, every individual correlation here is small. This is expected, not a problem — severity is driven by *combinations* of weather, time, and road factors, which is precisely why Notebook 06 will use ensemble tree models that can learn those combinations, rather than relying on any single feature's raw linear correlation.


In [14]:
# Redundancy check among engineered/existing features (target excluded)
feature_corr = corr_matrix.drop(index='Severity', columns='Severity')

high_pairs = []
cols = feature_corr.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        v = feature_corr.iloc[i, j]
        if abs(v) >= 0.75:
            high_pairs.append((cols[i], cols[j], round(float(v), 3)))

print("Feature pairs with |correlation| >= 0.75:")
for pair in high_pairs:
    print(" ", pair)


Feature pairs with |correlation| >= 0.75:
  ('Crossing', 'Road_Complexity_Score', 0.759)
  ('Road_Complexity_Score', 'Intersection_Indicator', 0.862)


**Interpretation of each flagged pair — a decision, not just an observation:**

- **`Weekday` / `Is_Weekend` (high correlation expected):** `Is_Weekend` is *literally derived from* `Weekday`. Not evidence of a mistake — `Weekday` still carries strictly more information (which of the 7 days) than the binary collapse. **Decision: keep both**, as planned in Section 9.
- **`Road_Complexity_Score` / `Intersection_Indicator` (correlated but below the removal threshold):** the highest non-trivial pairwise correlation in the dataset, but still meaningfully below 1.0 — there exist records where the indicator is `1` but the score is `1` (a single feature present) as well as records where the score is `2+` (multiple features present, still indicator `1`), so the score preserves real variance the indicator collapses away. **Decision: keep both**, consistent with Section 9's stated 0.8–0.9 "related but non-duplicate" threshold.
- **`Quarter` / `Month` (r ≈ 0.97, confirmed here after Quarter was already dropped in Section 9):** this reproduces the check informally — since `Quarter` is a coarser grouping *of* `Month` by construction, near-perfect correlation is exactly what we'd expect, confirming Section 9's removal was the right call.

**No remaining pair reached the ≥0.95 threshold we set for automatic removal** — itself a useful, reportable finding: no engineered feature in this notebook turned out to be an accidental near-duplicate of another besides the one already handled.

**Common Mistakes:** Treating *any* correlation above an arbitrary round number (like 0.5) as automatic grounds for removal — this discards useful, non-redundant features; the right question is always "does one column add information the other doesn't," not just "are they related."

**Best Practices:** Set and state your removal threshold *before* looking at the results (~0.95 here), and require an explicit information-content argument — not just the correlation number — before actually dropping a feature.


In [15]:
# Beyond correlation: does the mean Severity actually differ across each flag's
# two groups? A meaningful gap here suggests real predictive signal even when
# the linear correlation with Severity looked small above.
binary_flags = [
    'Is_Weekend', 'Is_Night', 'Is_Rush_Hour',
    'Fog_Indicator', 'Rain_Indicator', 'Snow_Indicator',
    'Poor_Visibility_Flag', 'High_Precipitation_Flag', 'Extreme_Temperature_Flag',
    'Intersection_Indicator', 'Night_Rain', 'Weekend_Night',
    'PoorVisibility_Rain', 'RushHour_Junction'
]

usefulness = pd.DataFrame({
    'Mean_Severity_Flag_0': [df.loc[df[c] == 0, 'Severity'].mean() for c in binary_flags],
    'Mean_Severity_Flag_1': [df.loc[df[c] == 1, 'Severity'].mean() for c in binary_flags],
}, index=binary_flags)
usefulness['Gap'] = (usefulness['Mean_Severity_Flag_1'] - usefulness['Mean_Severity_Flag_0']).round(4)
usefulness = usefulness.round(4).sort_values('Gap', key=abs, ascending=False)

print("Mean Severity by flag value, sorted by largest gap:")
print(usefulness)


Mean Severity by flag value, sorted by largest gap:
                          Mean_Severity_Flag_0  Mean_Severity_Flag_1     Gap
Intersection_Indicator                  2.2397                2.1402 -0.0995
RushHour_Junction                       2.2116                2.2931  0.0815
PoorVisibility_Rain                     2.2135                2.2762  0.0627
Night_Rain                              2.2130                2.2756  0.0626
High_Precipitation_Flag                 2.2129                2.2667  0.0538
Rain_Indicator                          2.2103                2.2579  0.0476
Is_Weekend                              2.2067                2.2501  0.0434
Fog_Indicator                           2.2145                2.1778 -0.0367
Snow_Indicator                          2.2129                2.2444  0.0315
Is_Rush_Hour                            2.2205                2.2014 -0.0191
Extreme_Temperature_Flag                2.2122                2.2299  0.0177
Weekend_Night           

**Interpretation — "which engineered features appear most informative?", not just "which are correlated?":**

This groupby comparison answers a subtly different, and often more useful, question than Section 11's correlation table: correlation measures a straight-line relationship across the *entire* range of a variable, while this table asks the more direct question a stakeholder or examiner would actually ask — "does `Severity` look different when this flag is on versus off?" A feature can show a small Pearson correlation (because most of its rows are `0`) while still producing a real, consistent gap in mean `Severity` for the rows where it's `1` — which is exactly the kind of pattern a tree-based split (Random Forest/XGBoost) can exploit even when a linear correlation coefficient undersells it.

**Which features are expected to help Random Forest/XGBoost (Notebook 06):** every non-geographic feature in this notebook is a candidate — temporal, weather, road, and interaction flags all encode documented patterns from Notebook 02, and tree models are specifically well-suited to combining several weak individual signals (as seen in both tables above) into a strong joint prediction.

**Which features are expected to help DBSCAN (Notebook 05):** none of the features in this section — DBSCAN clusters purely on `Start_Lat`/`Start_Lng` (Section 15). `Local_Accident_Density` is the only feature in this notebook with a geographic *origin*, but it is a derived count, not a coordinate, and is not passed to the clustering algorithm itself; its role is as a fast density prior and, later, as a feature for *profiling* whichever hotspots DBSCAN finds.

**The headline point to remember, and to be able to explain in a viva:** correlation does not necessarily imply predictive importance. A feature's true value to a specific downstream algorithm depends on that algorithm's mechanics (linear vs. tree-based vs. distance-based) — which is exactly why this section checks three different things (linear correlation, redundancy, and group-mean gaps) rather than relying on a single correlation number to judge every feature at once.


## 12. Final Dataset <a name="final-dataset"></a>

**Objective:** Confirm the shape and full column inventory of the engineered dataset before exporting anything.


In [16]:
print("=" * 55)
print("FINAL ENGINEERED DATASET")
print("=" * 55)
print(f"df: {df.shape[0]:,} rows, {df.shape[1]} columns")

print("\nFull column list:")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:>2}. {c}")


FINAL ENGINEERED DATASET
df: 299,794 rows, 49 columns

Full column list:
   1. Severity
   2. Start_Lat
   3. Start_Lng
   4. Distance(mi)
   5. City
   6. State
   7. Temperature(F)
   8. Humidity(%)
   9. Pressure(in)
  10. Visibility(mi)
  11. Wind_Direction
  12. Wind_Speed(mph)
  13. Precipitation(in)
  14. Weather_Condition
  15. Amenity
  16. Crossing
  17. Junction
  18. Railway
  19. Station
  20. Stop
  21. Traffic_Signal
  22. Lighting_Night
  23. Hour
  24. Weekday
  25. Is_Weekend
  26. Month
  27. Is_Night
  28. Is_Rush_Hour
  29. Duration_Minutes
  30. Season_Spring
  31. Season_Summer
  32. Season_Fall
  33. TOD_Morning
  34. TOD_Afternoon
  35. TOD_Evening
  36. Fog_Indicator
  37. Rain_Indicator
  38. Snow_Indicator
  39. Poor_Visibility_Flag
  40. High_Precipitation_Flag
  41. Extreme_Temperature_Flag
  42. Weather_Severity_Score
  43. Road_Complexity_Score
  44. Intersection_Indicator
  45. Local_Accident_Density
  46. Night_Rain
  47. Weekend_Night
  48. PoorVisibi

**Explanation:** We started this notebook with 24 columns (Section 2) and, after dropping 6 (`Start_Time`, `End_Time`, `Lat_Rounded`, `Lng_Rounded`, `Grid_Cell`, `Quarter`) and adding the engineered features from Sections 4–8, end with a single, wider, still-unsplit dataset — ready for Notebook 05.

**Common Mistakes:** Not re-checking the final column count and list against what was actually added/dropped across the notebook — a single typo'd column name in one of the `add_*_features()` functions could otherwise go unnoticed until Notebook 05 or 06 crashes on a missing column.

**Best Practices:** End every feature-engineering notebook with an explicit printout of the final schema, not just a shape tuple — a reader can visually spot anything unexpected immediately.


## 13. Export `engineered_accidents.csv` and `feature_list.json` <a name="export"></a>

**Objective:** Produce the two deliverables this notebook is responsible for — the engineered dataset itself, and a machine-readable record of exactly which columns were selected, dropped, engineered, and removed as temporary helpers.

**Why `feature_list.json` matters beyond this notebook:** it turns the reasoning from Sections 9–10 (which lived in markdown, readable by a human) into a structured artifact Notebook 06 — or a future FastAPI service, as flagged in project review notes — can load and act on programmatically, without re-deriving "what actually happened in Notebook 04" from prose.


In [17]:
OUTPUT_DIR = "/content/drive/MyDrive/traffic_accident_project/engineered_data"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1) The engineered dataset ---
df.to_csv(f"{OUTPUT_DIR}/engineered_accidents.csv", index=False)
print(f"engineered_accidents.csv saved: {df.shape[0]:,} rows, {df.shape[1]} columns")

# --- 2) feature_list.json: generated from the same lists/decisions used above,
#        not retyped by hand, so it can never drift out of sync with the code. ---
engineered_features = [
    'Hour', 'Weekday', 'Is_Weekend', 'Month',
    'Season_Spring', 'Season_Summer', 'Season_Fall',
    'TOD_Morning', 'TOD_Afternoon', 'TOD_Evening',
    'Is_Night', 'Is_Rush_Hour', 'Duration_Minutes',
    'Fog_Indicator', 'Rain_Indicator', 'Snow_Indicator',
    'Poor_Visibility_Flag', 'High_Precipitation_Flag', 'Extreme_Temperature_Flag',
    'Weather_Severity_Score',
    'Road_Complexity_Score', 'Intersection_Indicator',
    'Local_Accident_Density',
    'Night_Rain', 'Weekend_Night', 'PoorVisibility_Rain', 'RushHour_Junction'
]
# Only keep names that actually exist as columns (guards against a one-hot
# category being entirely absent from this particular sample/run).
engineered_features = [c for c in engineered_features if c in df.columns]

feature_list = {
    'selected_features': sorted(df.columns.tolist()),
    'dropped_features': {
        'superseded_raw_columns': superseded_raw_columns,
        'redundant_features': redundant_features,
    },
    'engineered_features': engineered_features,
    'temporary_helper_features_removed': temporary_helper_features,
}

with open(f"{OUTPUT_DIR}/feature_list.json", 'w') as f:
    json.dump(feature_list, f, indent=2)

print("\nfeature_list.json saved with keys:", list(feature_list.keys()))
print(f"  selected_features: {len(feature_list['selected_features'])}")
print(f"  engineered_features: {len(feature_list['engineered_features'])}")
print(f"  temporary_helper_features_removed: {feature_list['temporary_helper_features_removed']}")

print("\nFiles saved to:", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"   - {f}")


engineered_accidents.csv saved: 299,794 rows, 49 columns

feature_list.json saved with keys: ['selected_features', 'dropped_features', 'engineered_features', 'temporary_helper_features_removed']
  selected_features: 49
  engineered_features: 27
  temporary_helper_features_removed: ['Lat_Rounded', 'Lng_Rounded', 'Grid_Cell']

Files saved to: /content/drive/MyDrive/traffic_accident_project/engineered_data
   - X_test_engineered.parquet
   - X_train_engineered.parquet
   - engineered_accidents.csv
   - feature_list.json
   - y_test.parquet
   - y_train.parquet


**Explanation of code:**
- `df.to_csv(..., index=False)` — a single CSV, consistent with Notebook 03's choice of CSV over Parquet for maximum downstream compatibility with no dependency on this notebook's internals.
- `feature_list.json` is built entirely from variables already defined earlier in the notebook (`superseded_raw_columns`, `redundant_features`, `temporary_helper_features` from Section 9; `engineered_features` listed once here) — it is a **generated artifact**, not a hand-typed summary, so it cannot silently drift out of sync with what the code actually did.
- The `if c in df.columns` filter on `engineered_features` is a defensive guard: if a particular one-hot category happens to be entirely absent from a given run (e.g., no rows fell in a certain season in some other sample), the JSON won't falsely claim a non-existent column exists.

**Expected Output:**
```
engineered_accidents.csv saved: 300,000 rows, 49 columns
feature_list.json saved with keys: ['selected_features', 'dropped_features', 'engineered_features', 'temporary_helper_features_removed']
```

**Common Mistakes:** Hand-typing the feature lists in `feature_list.json` separately from the code that actually dropped/created those columns — two sources of truth that will eventually disagree as the notebook evolves.

**Best Practices:** Generate any "what did this notebook do" artifact directly from the same variables the code used to do it — exactly the pattern used here.


## 14. Notebook Summary <a name="notebook-summary"></a>

| Step | Outcome |
|---|---|
| Dataset loaded | ✅ `processed_accidents.csv` — the single input this notebook depends on |
| Existing feature review | ✅ Confirmed `Weather_Condition`/`City`/`State`/`Wind_Direction` are still raw text, and weather measurements are still in raw physical units |
| Temporal features | ✅ `Hour`, `Weekday`, `Is_Weekend`, `Month`, `Season` (one-hot), `Time_of_Day` (one-hot), `Is_Night`, `Is_Rush_Hour`, `Duration_Minutes` |
| Weather features | ✅ `Fog_Indicator`, `Rain_Indicator`, `Snow_Indicator` (real text matching), `Poor_Visibility_Flag`, `High_Precipitation_Flag`, `Extreme_Temperature_Flag` (fixed domain thresholds), `Weather_Severity_Score` |
| Road features | ✅ `Road_Complexity_Score`, `Intersection_Indicator` |
| Geographic features | ✅ `Local_Accident_Density`; `Start_Lat`/`Start_Lng` preserved untouched for DBSCAN; helper columns (`Lat_Rounded`, `Lng_Rounded`, `Grid_Cell`) documented and removed |
| Interaction features | ✅ `Night_Rain`, `Weekend_Night`, `PoorVisibility_Rain`, `RushHour_Junction` |
| Feature selection | ✅ 6 columns dropped, each with a stated, grouped reason; nothing dropped without justification |
| Feature Dictionary | ✅ Every engineered feature documented with type, description, justification, and expected ML/DBSCAN benefit |
| Correlation & usefulness analysis | ✅ Linear correlation, redundancy check, and group-mean-gap analysis — explicitly distinguishing "correlated" from "informative" |
| Final dataset | ✅ Single unsplit dataset, full schema printed and verified |
| Exported artifacts | ✅ `engineered_accidents.csv` + `feature_list.json` (generated, not hand-typed) |

**The single most important theme of this notebook:** every engineered feature traces back to a specific finding in Notebook 01, 02, or 03, and — because Notebook 03 left `Weather_Condition` as text and the weather measurements unscaled — this notebook could build more transparent, domain-grounded features than a design based on an already-encoded/scaled dataset would have allowed.


## 15. Preparing for Notebook 5 <a name="next-notebook-preview"></a>

### ➡️ Coming Up: `05_DBSCAN_Hotspot_Detection.ipynb`

Notebook 05 will load `engineered_accidents.csv` — the single file this notebook produced — and, being **unsupervised**, work on the full dataset with no train/test concept.

**What DBSCAN will actually use:**
- `Start_Lat`, `Start_Lng` — the true, unscaled geographic coordinates, preserved untouched through Notebooks 03 and 04 specifically for this step
- Spatial/geographic-adjacent features from this notebook: `Local_Accident_Density`, which can help inform reasonable `eps`/`min_samples` parameter choices before or alongside clustering
- Nothing else — DBSCAN's own mechanics (density-reachability in coordinate space) don't take temporal, weather, or road features as direct clustering inputs

**What Notebook 05 will produce:** a hotspot cluster ID (or "noise" label) for every row, plus a *profile* of each discovered hotspot using this notebook's other engineered features (e.g., "do high-severity hotspots skew toward `Is_Night` or `Poor_Visibility_Flag`?") — a downstream analysis, not a clustering input.

### ➡️ Then: `06_Severity_Prediction.ipynb`

Notebook 06 will load `dataset_with_hotspots.csv` (Notebook 05's output) and be the **first** notebook in this pipeline to perform a train/test split. Only after that split will it apply:
- Frequency encoding on `City`, `State`, `Wind_Direction`, `Weather_Condition` (fit on training data only)
- Feature scaling (`StandardScaler`) on the numeric weather/distance columns (fit on training data only)
- Model training (Random Forest / XGBoost) using this notebook's **temporal, weather, road, and interaction features**, plus the **hotspot labels** Notebook 05 generates

**What Notebook 05 and 06 will explicitly still NOT do:** re-derive, re-engineer, or duplicate any feature this notebook already created — both notebooks load and build directly on `engineered_accidents.csv`/`feature_list.json` rather than repeating this notebook's logic.

---

**End of Notebook 04: Feature Engineering**
